# Pipeline Profissional: YOLO + ByteTrack + SAM 3.1 + Dynamic Clustering

Neste notebook, faremos o **State-of-the-Art** do Sports Analytics:
1. **YOLO Pesado:** Achará jogadores e bola com altíssima precisão.
2. **ByteTrack:** Manterá um ID único para cada jogador enquanto ele corre na tela.
3. **SAM (Segment Anything):** Recortará perfeitamente a silhueta da camisa, eliminando grama, sol e sombras.
4. **K-Means Dinâmico:** Vai olhar para as silhuetas perfeitas e separar automaticamente em 2 times, sem precisarmos dizer quais são as cores!

**IMPORTANTE:** Rode este notebook no Google Colab com uma GPU ativada.

## 1. Instalação do Arsenal (Supervision, Ultralytics e SAM)

In [ ]:
!pip install ultralytics supervision scikit-learn -q
!pip install git+https://github.com/facebookresearch/segment-anything.git -q
!pip install easyocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 35.0 MB/s eta 0:00:00


## 2. Carregamento dos Modelos Pesados

In [ ]:
import torch
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
import supervision as sv

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando: {device}")

# 1. Carrega um YOLO muito mais inteligente (pré-treinado em milhões de imagens)
# Usaremos o modelo base mais pesado para ter a melhor precisão possível em humanos/bola
yolo_model = YOLO('yolov8x.pt')

# 2. Baixa e Carrega o SAM (Segment Anything)
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
sam.to(device=device)
sam_predictor = SamPredictor(sam)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Usando: cuda


## 3. Preparando o Rastreador (ByteTrack)
O ByteTrack garante que o "Jogador 5" continue sendo o "Jogador 5" mesmo que alguém passe na frente dele.

In [ ]:
tracker = sv.ByteTrack()

import cv2
import numpy as np
from sklearn.cluster import KMeans
import easyocr
from collections import defaultdict, Counter

print("Função de teste reservada.")


In [7]:
import cv2
import numpy as np
import supervision as sv
import torch
import easyocr
from collections import defaultdict, Counter
from sklearn.cluster import KMeans
from google.colab import files

print("Carregando motor de OCR...")
reader = easyocr.Reader(['en'], gpu=True)

VIDEO_PATH = "/content/futebol_teste.mp4"
OUTPUT_PATH = "futebol_rastreado.mp4"

ellipse_annotator = sv.EllipseAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)

# Nomes dos Times
TEAM_NAMES = {0: "Brasil", 1: "Marrocos"}

# Memórias do Tracker
tracker_jersey_memory = defaultdict(list)
tracker_team_memory = defaultdict(list)

def extract_dominant_color(image, k=2):
    pixels = image.reshape((-1, 3))
    pixels = np.float32(pixels)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    counts = np.bincount(labels.flatten())
    return np.uint8(centers[np.argmax(counts)])

def get_team_id(color):
    b, g, r = color
    if r > 100 and g > 100:
        return 0
    else:
        return 1

def process_frame(frame: np.ndarray, frame_index: int) -> np.ndarray:
    results = yolo_model(frame, classes=[0], conf=0.4, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    if len(detections) == 0:
        return frame

    sam_predictor.set_image(frame)
    input_boxes = torch.tensor(detections.xyxy, device=sam_predictor.device)
    transformed_boxes = sam_predictor.transform.apply_boxes_torch(input_boxes, frame.shape[:2])
    masks, _, _ = sam_predictor.predict_torch(
        point_coords=None, point_labels=None, boxes=transformed_boxes, multimask_output=False
    )
    detections.mask = masks.squeeze(1).cpu().numpy()

    final_labels = []
    for i in range(len(detections)):
        tracker_id = detections.tracker_id[i]
        x1, y1, x2, y2 = detections.xyxy[i].astype(int)

        player_crop = frame[y1:y2, x1:x2]

        if player_crop.size > 0:
            # OTIMIZAÇÃO 1: Só roda a extração de cor se tivermos menos de 10 amostras!
            if len(tracker_team_memory[tracker_id]) < 10:
                mask = detections.mask[i]
                mask_crop = mask[y1:y2, x1:x2]
                masked_player = cv2.bitwise_and(player_crop, player_crop, mask=mask_crop.astype(np.uint8))
                dominant_color = extract_dominant_color(masked_player)
                team_id = get_team_id(dominant_color)
                tracker_team_memory[tracker_id].append(team_id)

            # OTIMIZAÇÃO 2: Só roda o OCR se o jogador ainda não tiver 5 votos
            if len(tracker_jersey_memory[tracker_id]) < 5:
                if player_crop.shape[0] > 40 and player_crop.shape[1] > 20:
                    ocr_results = reader.readtext(player_crop, allowlist='0123456789')
                    for (bbox_ocr, text, prob) in ocr_results:
                        if prob > 0.3:
                            tracker_jersey_memory[tracker_id].append(text)

        # Votação do Time Histórico
        if len(tracker_team_memory[tracker_id]) > 0:
            final_team_id = Counter(tracker_team_memory[tracker_id]).most_common(1)[0][0]
            team_name = TEAM_NAMES.get(final_team_id, "Desconhecido")
        else:
            team_name = "Desconhecido"

        # Votação do Número Histórico
        if len(tracker_jersey_memory[tracker_id]) > 0:
            final_jersey = Counter(tracker_jersey_memory[tracker_id]).most_common(1)[0][0]
            label = f"Camisa {final_jersey} {team_name}"
        else:
            label = f"Jogador ({team_name})"

        final_labels.append(label)

    annotated_frame = ellipse_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=final_labels)

    return annotated_frame

sv.process_video(source_path=VIDEO_PATH, target_path=OUTPUT_PATH, callback=process_frame)
files.download(OUTPUT_PATH)


Carregando motor de OCR...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
